In [1]:
# =========================
# Config + imports
# =========================
import json
import os
import time
import warnings
import tempfile
from itertools import combinations
from pathlib import Path
from typing import Iterable, Mapping, Sequence

import joblib
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from IPython.display import display
from optuna.samplers import TPESampler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
# =========================
# Setup and configure
# =========================
DATA_PATH = Path("../data/train_dataset_v3.csv")

FOLDER_PATH = Path("results/random_forest_poisson")

COMBINATIONS_PATH = FOLDER_PATH / "0_feature_combinations.csv"
SCREENING_RESULTS_PATH = FOLDER_PATH / "1_screening_results.csv"
TOPK_PATH = FOLDER_PATH / "2_topk.csv"
TUNED_RESULTS_PATH = FOLDER_PATH / "3_tuned_results.csv"
BEST_CONFIG_PATH = FOLDER_PATH / "4_best_config.csv"
TEST_METRICS_PATH = FOLDER_PATH / "5_test_metrics.csv"
TEST_PREDICTIONS_PATH = FOLDER_PATH / "5_test_predictions.csv"
BEST_MODEL_PATH = FOLDER_PATH / "best_model.joblib"

TARGET_COL = "nielsen_total_volume"
TIME_COL = "continuous_week"

CAT_COLS = [
    "product_sku_code",
    "customer",
    "top_brand",
    "flavor_internal",
    "pack_type_internal",
    "material_medium_description",
]

NUM_COLS = [
    "promotion_indicator",
    "pack_size_internal",
    "units_per_package_internal",
    "price_per_litre",
    "week_sin",
    "week_cos",
    "continuous_week",
]

SERIES_COLS = ["product_sku_code", "customer"]

TEST_WEEK_RATIO = 0.20
N_SPLITS = 3
K = 5
TUNE_TRIALS = 10
RANDOM_STATE = 42
OPTUNA_SAMPLER_SEED = RANDOM_STATE
TUNING_OBJECTIVE = "mean_rmse"
TUNING_DIRECTION = "minimize"

MODEL_OBJECTIVE = "poisson"

RF_FIXED_PARAMS = {
    "criterion": MODEL_OBJECTIVE,
    "bootstrap": True,
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
}

match MODEL_OBJECTIVE:
    case "squared_error":
        pass
    case "poisson":
        pass
    case _:
        raise ValueError(
            f"{MODEL_OBJECTIVE} is not a valid model objective."
        )

MONOTONE_CONSTRAINTS = {
    # "price_per_litre": -1,
}


invalid_monotone_values = {
    feature: value
    for feature, value in MONOTONE_CONSTRAINTS.items()
    if int(value) not in (-1, 0, 1)
}
if invalid_monotone_values:
    raise ValueError(
        f"MONOTONE_CONSTRAINTS values must be in (-1, 0, 1): {invalid_monotone_values}"
    )
invalid_monotone_features = sorted(set(MONOTONE_CONSTRAINTS) - set(NUM_COLS))
if invalid_monotone_features:
    raise ValueError(
        f"MONOTONE_CONSTRAINTS contains features not in NUM_COLS: {invalid_monotone_features}"
    )


SCREENING_RF_PARAMS = {
    "n_estimators": 100,        # original 400
    "max_depth": 10,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "max_features": 1.0,
    "max_samples": 0.8,
    **RF_FIXED_PARAMS,
}

SCREENING_SORT_COLS = [
    "screening_mean_rmse",
    "screening_std_rmse",
    "screening_mean_rmsle",
    "feature_count",
]

SCREENING_SORT_TOL = [0.01, None, None, None]


TUNED_SORT_COLS = [
    "tuned_mean_rmse",
    "tuned_std_rmse",
    "tuned_mean_rmsle",
    "feature_count",
]

TUNED_SORT_TOL = [0.01, None, None, None]

EPS = 1e-12
SEASONALITY = 1
LATENCY_REPEATS = 5

In [3]:
# =========================
# Define all group cols
# =========================
MANDATORY_FEATURES = ["price_per_litre", "customer", "top_brand", "flavor_internal", "pack_type_internal", "pack_size_internal", "units_per_package_internal"]
# MANDATORY_FEATURES = ["price_per_litre"]

OPTIONAL_GROUPS = {
    # "sku": ["product_sku_code"],
    # "customer": ["customer"],                       #
    "promo": ["promotion_indicator"],               #
    # "brand": ["top_brand"],                         #
    # "flavor": ["flavor_internal"],                  #
    # "type": ["pack_type_internal"],                   #
    # "size": ["pack_size_internal"],                 #
    # "units": ["units_per_package_internal"],        #
    # "desc": ["material_medium_description"],
    "countweek": ["continuous_week"],
    "week": ["week_sin", "week_cos"],
}

NUM_FEATURE_POOL = set(NUM_COLS)
CAT_FEATURE_POOL = set(CAT_COLS)

print(f"Mandatory features: {MANDATORY_FEATURES}")
print(f"Optional groups: {list(OPTIONAL_GROUPS.keys())}")

Mandatory features: ['price_per_litre', 'customer', 'top_brand', 'flavor_internal', 'pack_type_internal', 'pack_size_internal', 'units_per_package_internal']
Optional groups: ['promo', 'countweek', 'week']


In [4]:
# =========================
# Helper methods
# =========================
def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def r2(y_true, y_pred):
    return float(r2_score(y_true, y_pred))


def rmsle(y_true, y_pred):
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)

    if np.any(y_true_arr < 0) or np.any(y_pred_arr < 0):
        warnings.warn(
            "RMSLE received negative values. They will be clipped to 0 before calculation.",
            RuntimeWarning
        )
        y_true_arr = np.clip(y_true_arr, 0.0, None)
        y_pred_arr = np.clip(y_pred_arr, 0.0, None)

    return float(np.sqrt(np.mean((np.log1p(y_pred_arr) - np.log1p(y_true_arr)) ** 2)))


def mape_pct(y_true, y_pred, eps=EPS):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100.0)


def smape_pct(y_true, y_pred, eps=EPS):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)) / 2.0, eps)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100.0)


def wmape_pct(y_true, y_pred, eps=EPS):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.sum(np.abs(y_true)), eps)
    return float(np.sum(np.abs(y_true - y_pred)) / denom * 100.0)


def panel_mase(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    y_pred,
    series_cols,
    target_col,
    time_col,
    seasonality=1,
    eps=EPS,
):
    train_sorted = train_df.sort_values(series_cols + [time_col]).copy()

    scale_df = (
        train_sorted
        .groupby(series_cols, sort=False)[target_col]
        .apply(
            lambda s: np.mean(np.abs(s.to_numpy()[seasonality:] - s.to_numpy()[:-seasonality]))
            if len(s) > seasonality else np.nan
        )
        .reset_index(name='_scale')
    )

    eval_df = test_df[series_cols + [target_col]].copy()
    eval_df['_y_pred'] = np.asarray(y_pred, dtype=float)
    eval_df = eval_df.merge(scale_df, on=series_cols, how='left')

    valid = eval_df['_scale'].notna() & (eval_df['_scale'] > eps)
    if not valid.any():
        return float('nan')

    scaled_abs_err = (
        np.abs(eval_df.loc[valid, target_col] - eval_df.loc[valid, '_y_pred'])
        / eval_df.loc[valid, '_scale']
    )
    return float(np.mean(scaled_abs_err))


def split_feature_types(
    features: Sequence[str],
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
) -> tuple[list[str], list[str]]:
    num_pool = set(num_feature_pool)
    cat_pool = set(cat_feature_pool)
    num_cols = [feature for feature in features if feature in num_pool]
    cat_cols = [feature for feature in features if feature in cat_pool]
    return num_cols, cat_cols


def make_preprocessor(num_cols: Sequence[str], cat_cols: Sequence[str]) -> ColumnTransformer:
    transformers = []

    if num_cols:
        transformers.append(("num", "passthrough", list(num_cols)))

    if cat_cols:
        transformers.append(
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                list(cat_cols),
            )
        )

    if not transformers:
        raise ValueError("At least one feature is required to build a preprocessor.")

    return ColumnTransformer(transformers=transformers, remainder="drop")


def get_monotone_constraints(X_fit, monotone_constraints: dict[str, int], num_cols: Sequence[str], cat_cols: Sequence[str]):
    preprocessor = make_preprocessor(num_cols, cat_cols)
    preprocessor.fit(X_fit)

    constraints = []
    seen_features = set()

    for feature_name in preprocessor.get_feature_names_out():
        if feature_name.startswith("num__"):
            raw_feature_name = feature_name.split("__", 1)[1]

            if raw_feature_name in monotone_constraints:
                seen_features.add(raw_feature_name)

            constraints.append(int(monotone_constraints.get(raw_feature_name, 0)))
        else:
            constraints.append(0)

    
    configured_features = {
        feature_name
        for feature_name, value in monotone_constraints.items()
        if int(value) != 0
    }
    missing_features = sorted(configured_features - seen_features)
    if missing_features:
        warnings.warn(
            f"Monotone-constrained features not found in transformed training features: {missing_features}"
        )

    return np.asarray(constraints, dtype=int)


def run_rf_cv(
    *,
    train_df: pd.DataFrame,
    target_col: str,
    feature_cols: Sequence[str],
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
    cv_splits: Sequence[tuple[np.ndarray, np.ndarray]],
    rf_params: Mapping[str, object],
    monotone_constraints: dict[str, int],
) -> dict[str, object]:
    num_cols, cat_cols = split_feature_types(
        feature_cols,
        num_feature_pool=num_feature_pool,
        cat_feature_pool=cat_feature_pool,
    )
    X_all = train_df.loc[:, list(feature_cols)].reset_index(drop=True)
    y_all = train_df[target_col].reset_index(drop=True)

    fold_rmse: list[float] = []
    fold_rmsle: list[float] = []
    fold_r2: list[float] = []
    fold_mape: list[float] = []
    fold_smape: list[float] = []
    fold_wmape: list[float] = []

    for train_idx, valid_idx in cv_splits:
        X_tr = X_all.iloc[train_idx]
        y_tr = y_all.iloc[train_idx]
        X_va = X_all.iloc[valid_idx]
        y_va = y_all.iloc[valid_idx]

        model_params = {
            **rf_params,
            "monotonic_cst": get_monotone_constraints(
                X_fit=X_tr,
                monotone_constraints=monotone_constraints,
                num_cols=num_cols,
                cat_cols=cat_cols,
            ),
        }

        pipe = Pipeline(
            steps=[
                ("prep", make_preprocessor(num_cols, cat_cols)),
                ("model", RandomForestRegressor(**model_params)),
            ]
        )

        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_va)
        y_pred = np.clip(y_pred, 0.0, None)

        fold_rmse.append(rmse(y_va, y_pred))
        fold_rmsle.append(rmsle(y_va, y_pred))
        fold_r2.append(r2(y_va, y_pred))
        fold_mape.append(mape_pct(y_va, y_pred))
        fold_smape.append(smape_pct(y_va, y_pred))
        fold_wmape.append(wmape_pct(y_va, y_pred))

    return {
        "fold_rmse": fold_rmse,
        "fold_rmsle": fold_rmsle,
        "fold_r2": fold_r2,
        "fold_mape": fold_mape,
        "fold_smape": fold_smape,
        "fold_wmape": fold_wmape,
        "mean_rmse": float(np.mean(fold_rmse)),
        "std_rmse": float(np.std(fold_rmse, ddof=0)),
        "mean_rmsle": float(np.mean(fold_rmsle)),
        "std_rmsle": float(np.std(fold_rmsle, ddof=0)),
        "mean_r2": float(np.mean(fold_r2)),
        "std_r2": float(np.std(fold_r2, ddof=0)),
        "mean_mape": float(np.mean(fold_mape)),
        "std_mape": float(np.std(fold_mape, ddof=0)),
        "mean_smape": float(np.mean(fold_smape)),
        "std_smape": float(np.std(fold_smape, ddof=0)),
        "mean_wmape": float(np.mean(fold_wmape)),
        "std_wmape": float(np.std(fold_wmape, ddof=0)),
    }


def sort_with_tolerances(
    df: pd.DataFrame,
    sort_cols: Sequence[str],
    ascending: Sequence[bool] | None = None,
    tolerances: Sequence[float | None] | None = None,
) -> pd.DataFrame:
    """
    Sort a dataframe using layered sort keys with optional relative near-tie tolerances.

    Each column in ``sort_cols`` defines one sorting layer. For a given layer:

    - if ``tolerances[i]`` is ``None``, rows are strictly ordered by
      ``sort_cols[i]`` and ties are passed to the next layer;
    - if ``tolerances[i]`` is a non-negative float, rows within the
      relative tolerance of the current best value at that layer are treated as
      belonging to the same tier and are then sorted by the next layer.

    Tolerances are only valid for numeric columns. Non-numeric columns must use
    ``None`` and typically act as final deterministic tie-breakers.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe to sort.
    sort_cols : Sequence[str]
        Ordered list of sorting columns from highest to lowest priority.
    ascending : Sequence[bool] | None, default None
        Sort direction for each sorting column. Must have the same length as
        ``sort_cols`` when provided. If ``None``, all layers are sorted in
        ascending order.
    tolerances : Sequence[float | None] default None
        Relative tolerance for each sorting column. Must have the same length as
        ``sort_cols``. Use ``None`` for strict comparison at that layer.

    Returns
    -------
    pd.DataFrame
        A reordered dataframe containing all input rows, sorted according to the
        layered sorting and tolerance rules.

    Raises
    ------
    ValueError
        If input lengths do not match, required columns are missing, ascending
        contains non-boolean values, a tolerance is negative, or a tolerance is
        assigned to a non-numeric column.
    """
    if not sort_cols:
        raise ValueError("sort_cols must contain at least one column.")

    tolerances = [None] * len(sort_cols) if tolerances is None else list(tolerances)

    if len(sort_cols) != len(tolerances):
        raise ValueError("sort_cols and tolerances must have the same length.")

    if ascending is None:
        ascending = [True] * len(sort_cols)
    else:
        ascending = list(ascending)

    if len(sort_cols) != len(ascending):
        raise ValueError("sort_cols and ascending must have the same length.")

    if any(not isinstance(x, bool) for x in ascending):
        raise ValueError("ascending must contain only True/False values.")

    missing_cols = [col for col in sort_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing sort columns: {missing_cols}")

    groups = [df.copy()]

    for col, tol, asc in zip(sort_cols, tolerances, ascending):
        next_groups = []

        for group in groups:
            if group.empty or len(group) == 1:
                next_groups.append(group)
                continue

            ordered = group.sort_values(by=col, ascending=asc, kind="stable")

            if tol is None:
                for _, subgroup in ordered.groupby(col, sort=False, dropna=False):
                    next_groups.append(subgroup.copy())
                continue

            if tol < 0:
                raise ValueError(f"Tolerance for '{col}' must be >= 0.")

            if not pd.api.types.is_numeric_dtype(ordered[col]):
                raise ValueError(f"Column '{col}' is non-numeric, so its tolerance must be None.")

            remaining = ordered.copy()

            while not remaining.empty:
                best_value = float(remaining.iloc[0][col])
                delta = max(abs(best_value), 1e-12) * tol

                if asc:
                    mask = remaining[col] <= best_value + delta
                else:
                    mask = remaining[col] >= best_value - delta

                tier = remaining.loc[mask].copy()
                next_groups.append(tier)
                remaining = remaining.loc[~mask].copy()

        groups = next_groups

    sorted_df = pd.concat(groups, axis=0, ignore_index=True)
    return sorted_df.reset_index(drop=True)



In [5]:
# =========================
# Load dataset
# =========================
FOLDER_PATH.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH).copy()

print(f"Loaded rows: {len(df):,}")
print(f"Loaded columns: {len(df.columns):,}")
DATA_PATH

Loaded rows: 92,777
Loaded columns: 20


WindowsPath('../data/train_dataset_v3.csv')

In [6]:
# =========================
# Validate dataset
# =========================
required_cols = list(dict.fromkeys(CAT_COLS + NUM_COLS + SERIES_COLS + [TARGET_COL, TIME_COL]))
missing_cols = sorted(set(required_cols) - set(df.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if df.empty:
    raise ValueError("The dataset is empty.")

_ = pd.to_numeric(df[TIME_COL], errors="raise")
target_series = pd.to_numeric(df[TARGET_COL], errors="raise")
target_min = float(target_series.min())

if MODEL_OBJECTIVE == "poisson" and target_min < 0:
    raise ValueError(
        f"MODEL_OBJECTIVE='poisson' requires non-negative '{TARGET_COL}'. Observed min={target_min}."
    )

poisson_target_validation_passed = MODEL_OBJECTIVE != "poisson" or target_min >= 0

print(f"Rows: {len(df):,}")
print(f"Unique weeks: {df[TIME_COL].nunique():,}")
print(f"Target: {TARGET_COL}")
print(f"Model objective: {MODEL_OBJECTIVE}")
print(f"Poisson target validation passed: {poisson_target_validation_passed}")
print(f"Required columns validated: {len(required_cols)}")

Rows: 92,777
Unique weeks: 145
Target: nielsen_total_volume
Model objective: poisson
Poisson target validation passed: True
Required columns validated: 14


In [7]:
# =========================
# Time split train_df, test_df
# =========================
df = df.sort_values([TIME_COL] + SERIES_COLS).reset_index(drop=True).copy()
weeks = np.sort(df[TIME_COL].dropna().astype(int).unique())

if len(weeks) < 2:
    raise ValueError("Need at least 2 unique weeks for a train/test split.")

n_test_weeks = max(1, int(round(len(weeks) * TEST_WEEK_RATIO)))
n_test_weeks = min(n_test_weeks, len(weeks) - 1)
cut_week = int(weeks[-n_test_weeks])

train_df = df[df[TIME_COL] < cut_week].copy()
test_df = df[df[TIME_COL] >= cut_week].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Train or test split is empty. Check TEST_WEEK_RATIO or data coverage.")

if int(train_df[TIME_COL].max()) >= int(test_df[TIME_COL].min()):
    raise ValueError("Time split is invalid: train overlaps test.")

print(f"Cut week: {cut_week}")
print(
    f"Train weeks: {int(train_df[TIME_COL].min())} -> {int(train_df[TIME_COL].max())} "
    f"({train_df[TIME_COL].nunique()} weeks)"
)
print(
    f"Test weeks:  {int(test_df[TIME_COL].min())} -> {int(test_df[TIME_COL].max())} "
    f"({test_df[TIME_COL].nunique()} weeks)"
)
print(f"Train rows: {len(train_df):,}")
print(f"Test rows:  {len(test_df):,}")

Cut week: 118
Train weeks: 0 -> 117 (116 weeks)
Test weeks:  118 -> 146 (29 weeks)
Train rows: 73,887
Test rows:  18,890


In [8]:
# =========================
# Define shared time-based CV on train_df
# =========================
# methods
def make_expanding_time_splits(
    df: pd.DataFrame,
    time_col: str,
    n_splits: int = 3,
) -> list[tuple[np.ndarray, np.ndarray]]:
    unique_periods = np.sort(df[time_col].dropna().astype(int).unique())

    if len(unique_periods) < n_splits + 1:
        raise ValueError(
            f"Not enough unique periods in '{time_col}' for {n_splits} expanding folds."
        )

    period_blocks = np.array_split(unique_periods, n_splits + 1)

    folds: list[tuple[np.ndarray, np.ndarray]] = []
    for fold_idx in range(1, n_splits + 1):
        train_periods = np.concatenate(period_blocks[:fold_idx])
        valid_periods = period_blocks[fold_idx]

        train_idx = np.flatnonzero(df[time_col].isin(train_periods).to_numpy())
        valid_idx = np.flatnonzero(df[time_col].isin(valid_periods).to_numpy())

        if len(train_idx) == 0 or len(valid_idx) == 0:
            raise ValueError(
                "Generated an empty train or validation fold. "
                "Check the time column coverage and n_splits."
            )

        folds.append((train_idx, valid_idx))

    return folds


def build_fold_summary(
    df: pd.DataFrame,
    time_col: str,
    cv_splits: Sequence[tuple[np.ndarray, np.ndarray]],
) -> pd.DataFrame:
    rows = []

    for fold_idx, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
        train_periods = df.iloc[train_idx][time_col].astype(int)
        valid_periods = df.iloc[valid_idx][time_col].astype(int)

        rows.append(
            {
                "fold": fold_idx,
                "train_rows": int(len(train_idx)),
                "valid_rows": int(len(valid_idx)),
                "train_period_min": int(train_periods.min()),
                "train_period_max": int(train_periods.max()),
                "valid_period_min": int(valid_periods.min()),
                "valid_period_max": int(valid_periods.max()),
            }
        )

    return pd.DataFrame(rows)


shared_train_df = train_df.sort_values([TIME_COL] + SERIES_COLS).reset_index(drop=True).copy()
shared_cv_splits = make_expanding_time_splits(shared_train_df, time_col=TIME_COL, n_splits=N_SPLITS)
shared_fold_summary_df = build_fold_summary(shared_train_df, time_col=TIME_COL, cv_splits=shared_cv_splits)

print(f"Shared CV folds: {len(shared_cv_splits)}")
display(shared_fold_summary_df)

Shared CV folds: 3


,fold,train_rows,valid_rows,train_period_min,train_period_max,valid_period_min,valid_period_max
0,1,17836,18598,0,28,29,58
1,2,36434,18874,0,58,59,87
2,3,55308,18579,0,87,88,117


In [9]:
# =========================
# Get all valid group combinations
# =========================
# methods
def validate_feature_groups(
    mandatory_features: Sequence[str],
    optional_groups: Mapping[str, Sequence[str]],
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
) -> None:
    num_pool = set(num_feature_pool)
    cat_pool = set(cat_feature_pool)
    known_features = num_pool | cat_pool

    unknown_mandatory = sorted(set(mandatory_features) - known_features)
    if unknown_mandatory:
        raise ValueError(
            f"Mandatory features are not in the numeric/categorical pools: {unknown_mandatory}"
        )

    seen_optional: dict[str, str] = {}
    for group_name, group_features in optional_groups.items():
        if not group_features:
            raise ValueError(f"Optional group '{group_name}' is empty.")

        unknown_group = sorted(set(group_features) - known_features)
        if unknown_group:
            raise ValueError(f"Group '{group_name}' contains unknown features: {unknown_group}")

        overlap_mandatory = sorted(set(group_features) & set(mandatory_features))
        if overlap_mandatory:
            raise ValueError(
                f"Group '{group_name}' overlaps with mandatory features: {overlap_mandatory}"
            )

        for feature_name in group_features:
            previous_group = seen_optional.get(feature_name)
            if previous_group is not None:
                raise ValueError(
                    f"Feature '{feature_name}' appears in both '{previous_group}' and '{group_name}'."
                )
            seen_optional[feature_name] = group_name


def get_feature_combinations(
    *,
    mandatory_features: Sequence[str],
    optional_groups: Mapping[str, Sequence[str]],
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
) -> pd.DataFrame:
    validate_feature_groups(
        mandatory_features=mandatory_features,
        optional_groups=optional_groups,
        num_feature_pool=num_feature_pool,
        cat_feature_pool=cat_feature_pool,
    )

    group_names = list(optional_groups.keys())
    mandatory_label = " + ".join(mandatory_features)
    rows: list[dict[str, object]] = []

    for group_count in range(len(group_names) + 1):
        for selected_group_names in combinations(group_names, group_count):
            ordered_features = list(dict.fromkeys(mandatory_features))
            for group_name in selected_group_names:
                ordered_features.extend(optional_groups[group_name])
            ordered_features = list(dict.fromkeys(ordered_features))

            groups = f"{mandatory_label}_only"
            if selected_group_names:
                groups = mandatory_label + " + " + " + ".join(selected_group_names)

            rows.append(
                {
                    "groups": groups,
                    "feature_list_json": json.dumps(ordered_features, ensure_ascii=True),
                    "feature_count": len(ordered_features),
                }
            )

    return pd.DataFrame(rows).reset_index(drop=True)


# execution code
combinations_df = get_feature_combinations(
    mandatory_features=MANDATORY_FEATURES,
    optional_groups=OPTIONAL_GROUPS,
    num_feature_pool=NUM_FEATURE_POOL,
    cat_feature_pool=CAT_FEATURE_POOL,
)

combinations_df.insert(0, "combination_id", range(1, len(combinations_df) + 1))

print(f"Total valid combinations: {len(combinations_df)}")
display(combinations_df)

Total valid combinations: 8


,combination_id,groups,feature_list_json,feature_count
0,1,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",7
1,2,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8
2,3,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8
3,4,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9
4,5,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9
5,6,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",10
6,7,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",10
7,8,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",11


In [10]:
# =========================
# Stage 0 - save feature combination with index
# =========================
combinations_df.to_csv(COMBINATIONS_PATH, index=False)
print(f"Saved: {COMBINATIONS_PATH}")

Saved: results\random_forest_poisson\0_feature_combinations.csv


In [11]:
# =========================
# Stage 1 - run baseline screening
# =========================
# methods
def evaluate_feature(
    *,
    combination_id: int,
    groups: str,
    feature_cols: Sequence[str],
    train_df: pd.DataFrame,
    target_col: str,
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
    cv_splits: Sequence[tuple[np.ndarray, np.ndarray]],
    rf_params: Mapping[str, object],
    monotone_constraints: dict[str, int],
) -> dict[str, object]:
    cv_results = run_rf_cv(
        train_df=train_df,
        target_col=target_col,
        feature_cols=feature_cols,
        num_feature_pool=num_feature_pool,
        cat_feature_pool=cat_feature_pool,
        cv_splits=cv_splits,
        rf_params=rf_params,
        monotone_constraints=monotone_constraints,
    )

    row: dict[str, object] = {
        "combination_id": combination_id,
        "groups": groups,
        "feature_list_json": json.dumps(list(feature_cols), ensure_ascii=True),
        "feature_count": len(feature_cols),

        "screening_mean_rmse": float(cv_results["mean_rmse"]),
        "screening_std_rmse": float(cv_results["std_rmse"]),
        "screening_mean_rmsle": float(cv_results["mean_rmsle"]),
        "screening_std_rmsle": float(cv_results["std_rmsle"]),
        "screening_mean_r2": float(cv_results["mean_r2"]),
        "screening_std_r2": float(cv_results["std_r2"]),

        "screening_mean_mape": float(cv_results["mean_mape"]),
        "screening_std_mape": float(cv_results["std_mape"]),
        "screening_mean_smape": float(cv_results["mean_smape"]),
        "screening_std_smape": float(cv_results["std_smape"]),
        "screening_mean_wmape": float(cv_results["mean_wmape"]),
        "screening_std_wmape": float(cv_results["std_wmape"]),
    }

    for fold_idx in range(1, len(cv_splits) + 1):
        row[f"screening_fold_{fold_idx}_rmse"] = float(cv_results["fold_rmse"][fold_idx - 1])
        row[f"screening_fold_{fold_idx}_rmsle"] = float(cv_results["fold_rmsle"][fold_idx - 1])
        row[f"screening_fold_{fold_idx}_r2"] = float(cv_results["fold_r2"][fold_idx - 1])
        row[f"screening_fold_{fold_idx}_mape"] = float(cv_results["fold_mape"][fold_idx - 1])
        row[f"screening_fold_{fold_idx}_smape"] = float(cv_results["fold_smape"][fold_idx - 1])
        row[f"screening_fold_{fold_idx}_wmape"] = float(cv_results["fold_wmape"][fold_idx - 1])

    return row


def run_feature_screening(
    *,
    train_df: pd.DataFrame,
    target_col: str,
    combinations_df: pd.DataFrame,
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
    cv_splits: Sequence[tuple[np.ndarray, np.ndarray]],
    rf_params: Mapping[str, object],
    monotone_constraints: dict[str, int],
) -> pd.DataFrame:
    rows = []
    for _, combination_row in combinations_df.iterrows():
        rows.append(
            evaluate_feature(
                combination_id=combination_row["combination_id"],
                groups=str(combination_row['groups']),
                feature_cols=json.loads(combination_row["feature_list_json"]),
                train_df=train_df,
                target_col=target_col,
                num_feature_pool=num_feature_pool,
                cat_feature_pool=cat_feature_pool,
                cv_splits=cv_splits,
                rf_params=rf_params,
                monotone_constraints=monotone_constraints,
            )
        )

    results_df = pd.DataFrame(rows)

    return results_df.reset_index(drop=True)


# execution code
# load combination_df
combinations_df = pd.read_csv(COMBINATIONS_PATH)


screening_results_df = run_feature_screening(
    train_df=shared_train_df,
    target_col=TARGET_COL,
    combinations_df=combinations_df,
    num_feature_pool=NUM_FEATURE_POOL,
    cat_feature_pool=CAT_FEATURE_POOL,
    cv_splits=shared_cv_splits,
    rf_params=SCREENING_RF_PARAMS,
    monotone_constraints=MONOTONE_CONSTRAINTS,
)

print(f"Screened combinations: {len(screening_results_df)}")
display(screening_results_df)


Screened combinations: 8


,combination_id,groups,feature_list_json,feature_count,screening_mean_rmse,screening_std_rmse,screening_mean_rmsle,screening_std_rmsle,screening_mean_r2,screening_std_r2,...,screening_fold_2_r2,screening_fold_2_mape,screening_fold_2_smape,screening_fold_2_wmape,screening_fold_3_rmse,screening_fold_3_rmsle,screening_fold_3_r2,screening_fold_3_mape,screening_fold_3_smape,screening_fold_3_wmape
0,1,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",7,7656.652171,1436.489061,1.592344,0.109367,0.838300,0.048514,...,0.876291,6655.020693,59.630024,27.055156,6640.155303,1.470615,0.868781,3041.143578,56.852625,32.405026
1,2,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,7093.322541,1221.554691,1.551657,0.101322,0.861280,0.037655,...,0.901625,5163.890989,59.240515,25.346152,6578.452117,1.438025,0.871208,2665.402940,56.533108,31.242400
2,3,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,7335.416437,1543.753736,1.570690,0.096662,0.850692,0.051804,...,0.895765,4952.192145,59.435614,24.835766,6398.066927,1.454865,0.878174,2968.192958,56.207112,32.378705
3,4,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9,7635.458328,1976.076187,1.629612,0.137290,0.836252,0.072216,...,0.878107,9130.547420,60.423016,28.291642,5911.437518,1.482703,0.896001,3147.303936,56.683623,30.254614
4,5,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9,6847.271508,1423.307048,1.534158,0.098305,0.869954,0.044445,...,0.909762,4443.579889,58.967734,24.406179,6019.242502,1.414436,0.892174,2605.522278,56.067175,30.410310
5,6,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",10,7273.366766,1908.670166,1.586938,0.128177,0.851142,0.067136,...,0.898768,6931.844654,60.064523,26.913729,5841.137072,1.446133,0.898460,2709.867573,56.347401,29.429500
6,7,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",10,7128.411147,1774.391834,1.591095,0.114487,0.857663,0.060305,...,0.899298,6074.065576,59.419566,25.148692,5758.772743,1.454461,0.901304,2935.324448,55.571295,29.235536
7,8,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",11,6817.094996,1664.394701,1.552534,0.109572,0.869920,0.053955,...,0.912331,5147.925490,58.993031,25.019566,5689.979810,1.418218,0.903648,2605.376127,55.551440,28.673227


In [12]:
# =========================
# Stage 1 - save screening outputs
# =========================
screening_results_df.to_csv(SCREENING_RESULTS_PATH, index=False)
print(f"Saved: {SCREENING_RESULTS_PATH}")

Saved: results\random_forest_poisson\1_screening_results.csv


In [13]:
# =========================
# Stage 2 - load screening results, sort, select top k
# =========================
screening_results_df = pd.read_csv(SCREENING_RESULTS_PATH)

# sort by rule
sorted_screening_results_df = sort_with_tolerances(
    df=screening_results_df,
    sort_cols=SCREENING_SORT_COLS,
    tolerances=SCREENING_SORT_TOL,
).reset_index(drop=True)

display(sorted_screening_results_df)

topk_df = sorted_screening_results_df.head(K).reset_index(drop=True)

print(f"Top-K rows: {len(topk_df)}")
display(topk_df)

,combination_id,groups,feature_list_json,feature_count,screening_mean_rmse,screening_std_rmse,screening_mean_rmsle,screening_std_rmsle,screening_mean_r2,screening_std_r2,...,screening_fold_2_r2,screening_fold_2_mape,screening_fold_2_smape,screening_fold_2_wmape,screening_fold_3_rmse,screening_fold_3_rmsle,screening_fold_3_r2,screening_fold_3_mape,screening_fold_3_smape,screening_fold_3_wmape
0,5,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9,6847.271508,1423.307048,1.534158,0.098305,0.869954,0.044445,...,0.909762,4443.579889,58.967734,24.406179,6019.242502,1.414436,0.892174,2605.522278,56.067175,30.410310
1,8,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",11,6817.094996,1664.394701,1.552534,0.109572,0.869920,0.053955,...,0.912331,5147.925490,58.993031,25.019566,5689.979810,1.418218,0.903648,2605.376127,55.551440,28.673227
2,2,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,7093.322541,1221.554691,1.551657,0.101322,0.861280,0.037655,...,0.901625,5163.890989,59.240515,25.346152,6578.452117,1.438025,0.871208,2665.402940,56.533108,31.242400
3,7,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",10,7128.411147,1774.391834,1.591095,0.114487,0.857663,0.060305,...,0.899298,6074.065576,59.419566,25.148692,5758.772743,1.454461,0.901304,2935.324448,55.571295,29.235536
4,3,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,7335.416437,1543.753736,1.570690,0.096662,0.850692,0.051804,...,0.895765,4952.192145,59.435614,24.835766,6398.066927,1.454865,0.878174,2968.192958,56.207112,32.378705
5,6,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",10,7273.366766,1908.670166,1.586938,0.128177,0.851142,0.067136,...,0.898768,6931.844654,60.064523,26.913729,5841.137072,1.446133,0.898460,2709.867573,56.347401,29.429500
6,1,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",7,7656.652171,1436.489061,1.592344,0.109367,0.838300,0.048514,...,0.876291,6655.020693,59.630024,27.055156,6640.155303,1.470615,0.868781,3041.143578,56.852625,32.405026
7,4,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9,7635.458328,1976.076187,1.629612,0.137290,0.836252,0.072216,...,0.878107,9130.547420,60.423016,28.291642,5911.437518,1.482703,0.896001,3147.303936,56.683623,30.254614


Top-K rows: 5


,combination_id,groups,feature_list_json,feature_count,screening_mean_rmse,screening_std_rmse,screening_mean_rmsle,screening_std_rmsle,screening_mean_r2,screening_std_r2,...,screening_fold_2_r2,screening_fold_2_mape,screening_fold_2_smape,screening_fold_2_wmape,screening_fold_3_rmse,screening_fold_3_rmsle,screening_fold_3_r2,screening_fold_3_mape,screening_fold_3_smape,screening_fold_3_wmape
0,5,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9,6847.271508,1423.307048,1.534158,0.098305,0.869954,0.044445,...,0.909762,4443.579889,58.967734,24.406179,6019.242502,1.414436,0.892174,2605.522278,56.067175,30.410310
1,8,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",11,6817.094996,1664.394701,1.552534,0.109572,0.869920,0.053955,...,0.912331,5147.925490,58.993031,25.019566,5689.979810,1.418218,0.903648,2605.376127,55.551440,28.673227
2,2,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,7093.322541,1221.554691,1.551657,0.101322,0.861280,0.037655,...,0.901625,5163.890989,59.240515,25.346152,6578.452117,1.438025,0.871208,2665.402940,56.533108,31.242400
3,7,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",10,7128.411147,1774.391834,1.591095,0.114487,0.857663,0.060305,...,0.899298,6074.065576,59.419566,25.148692,5758.772743,1.454461,0.901304,2935.324448,55.571295,29.235536
4,3,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,7335.416437,1543.753736,1.570690,0.096662,0.850692,0.051804,...,0.895765,4952.192145,59.435614,24.835766,6398.066927,1.454865,0.878174,2968.192958,56.207112,32.378705


In [14]:
# =========================
# Stage 2 - save topk
# =========================
topk_df.to_csv(TOPK_PATH, index=False)
print(f"Saved: {TOPK_PATH}")

Saved: results\random_forest_poisson\2_topk.csv


In [15]:
# =========================
# Stage 3 - load topk and run Optuna tuning
# =========================
# method
def build_tuning_objective(
    *,
    train_df: pd.DataFrame,
    target_col: str,
    feature_cols: Sequence[str],
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
    cv_splits: Sequence[tuple[np.ndarray, np.ndarray]],
    rf_fixed_params: Mapping[str, object],
    tuning_objective: str,
    monotone_constraints: dict[str, int],
):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 600),
            "max_depth": trial.suggest_int("max_depth", 6, 16),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "max_features": trial.suggest_float("max_features", 0.3, 1.0),
            "max_samples": trial.suggest_float("max_samples", 0.6, 1.0),
            **rf_fixed_params,
        }

        cv_results = run_rf_cv(
            train_df=train_df,
            target_col=target_col,
            feature_cols=feature_cols,
            num_feature_pool=num_feature_pool,
            cat_feature_pool=cat_feature_pool,
            cv_splits=cv_splits,
            rf_params=params,
            monotone_constraints=monotone_constraints,
        )

        trial.set_user_attr("mean_rmse", float(cv_results["mean_rmse"]))
        trial.set_user_attr("std_rmse", float(cv_results["std_rmse"]))
        trial.set_user_attr("mean_rmsle", float(cv_results["mean_rmsle"]))
        trial.set_user_attr("std_rmsle", float(cv_results["std_rmsle"]))
        trial.set_user_attr("mean_r2", float(cv_results["mean_r2"]))
        trial.set_user_attr("std_r2", float(cv_results["std_r2"]))

        trial.set_user_attr("mean_mape", float(cv_results["mean_mape"]))
        trial.set_user_attr("std_mape", float(cv_results["std_mape"]))
        trial.set_user_attr("mean_smape", float(cv_results["mean_smape"]))
        trial.set_user_attr("std_smape", float(cv_results["std_smape"]))
        trial.set_user_attr("mean_wmape", float(cv_results["mean_wmape"]))
        trial.set_user_attr("std_wmape", float(cv_results["std_wmape"]))

        for fold_idx, value in enumerate(cv_results["fold_rmse"], start=1):
            trial.set_user_attr(f"fold_{fold_idx}_rmse", float(value))

        for fold_idx, value in enumerate(cv_results["fold_rmsle"], start=1):
            trial.set_user_attr(f"fold_{fold_idx}_rmsle", float(value))

        for fold_idx, value in enumerate(cv_results["fold_r2"], start=1):
            trial.set_user_attr(f"fold_{fold_idx}_r2", float(value))

        for fold_idx, value in enumerate(cv_results["fold_mape"], start=1):
            trial.set_user_attr(f"fold_{fold_idx}_mape", float(value))

        for fold_idx, value in enumerate(cv_results["fold_smape"], start=1):
            trial.set_user_attr(f"fold_{fold_idx}_smape", float(value))

        for fold_idx, value in enumerate(cv_results["fold_wmape"], start=1):
            trial.set_user_attr(f"fold_{fold_idx}_wmape", float(value))

        return float(cv_results[tuning_objective])

    return objective


def evaluate_feature_combination_for_tuning(
    *,
    combination_id: int,
    groups: str,
    feature_cols: Sequence[str],
    train_df: pd.DataFrame,
    target_col: str,
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
    cv_splits: Sequence[tuple[np.ndarray, np.ndarray]],
    rf_fixed_params: Mapping[str, object],
    tuning_objective: str,
    tuning_direction: str,
    sampler_seed: int,
    tune_trials: int,
    monotone_constraints: dict[str, int]
) -> dict[str, object]:
    study = optuna.create_study(
        direction=tuning_direction,
        sampler=TPESampler(seed=sampler_seed),
    )

    t0 = time.perf_counter()
    study.optimize(
        build_tuning_objective(
            train_df=train_df,
            target_col=target_col,
            feature_cols=feature_cols,
            num_feature_pool=num_feature_pool,
            cat_feature_pool=cat_feature_pool,
            cv_splits=cv_splits,
            rf_fixed_params=rf_fixed_params,
            tuning_objective=tuning_objective,
            monotone_constraints=monotone_constraints,
        ),
        n_trials=tune_trials,
        show_progress_bar=False,
    )
    tuning_time_s = time.perf_counter() - t0

    best_trial = study.best_trial
    best_params = {
        **best_trial.params,
        **dict(rf_fixed_params),
    }
    row = {
        "combination_id": combination_id,
        "groups": groups,
        "feature_list_json": json.dumps(list(feature_cols), ensure_ascii=True),
        "feature_count": len(feature_cols),

        "tuned_mean_rmse": float(best_trial.user_attrs["mean_rmse"]),
        "tuned_std_rmse": float(best_trial.user_attrs["std_rmse"]),
        "tuned_mean_rmsle": float(best_trial.user_attrs["mean_rmsle"]),
        "tuned_std_rmsle": float(best_trial.user_attrs["std_rmsle"]),
        "tuned_mean_r2": float(best_trial.user_attrs["mean_r2"]),
        "tuned_std_r2": float(best_trial.user_attrs["std_r2"]),
        "tuned_mean_mape": float(best_trial.user_attrs["mean_mape"]),
        "tuned_std_mape": float(best_trial.user_attrs["std_mape"]),
        "tuned_mean_smape": float(best_trial.user_attrs["mean_smape"]),
        "tuned_std_smape": float(best_trial.user_attrs["std_smape"]),
        "tuned_mean_wmape": float(best_trial.user_attrs["mean_wmape"]),
        "tuned_std_wmape": float(best_trial.user_attrs["std_wmape"]),

        "tuning_trials": int(tune_trials),
        "tuning_time_s": float(tuning_time_s),
        "best_params_json": json.dumps(best_params, ensure_ascii=True),
    }

    for fold_idx in range(1, len(cv_splits) + 1):
        row[f"tuned_fold_{fold_idx}_rmse"] = float(best_trial.user_attrs[f"fold_{fold_idx}_rmse"])
        row[f"tuned_fold_{fold_idx}_rmsle"] = float(best_trial.user_attrs[f"fold_{fold_idx}_rmsle"])
        row[f"tuned_fold_{fold_idx}_r2"] = float(best_trial.user_attrs[f"fold_{fold_idx}_r2"])
        row[f"tuned_fold_{fold_idx}_mape"] = float(best_trial.user_attrs[f"fold_{fold_idx}_mape"])
        row[f"tuned_fold_{fold_idx}_smape"] = float(best_trial.user_attrs[f"fold_{fold_idx}_smape"])
        row[f"tuned_fold_{fold_idx}_wmape"] = float(best_trial.user_attrs[f"fold_{fold_idx}_wmape"])

    return row


def run_feature_tuning(
    *,
    train_df: pd.DataFrame,
    target_col: str,
    combinations_df: pd.DataFrame,
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
    cv_splits: Sequence[tuple[np.ndarray, np.ndarray]],
    rf_fixed_params: Mapping[str, object],
    tuning_objective: str,
    tuning_direction: str,
    sampler_seed: int,
    tune_trials: int,
    monotone_constraints: dict[str, int],
):
    rows = []
    for _, combination_row in combinations_df.iterrows():
        rows.append(
            evaluate_feature_combination_for_tuning(  
                combination_id=combination_row["combination_id"],
                groups=str(combination_row["groups"]),
                feature_cols=json.loads(combination_row["feature_list_json"]),
                train_df=train_df,
                target_col=target_col,
                num_feature_pool=num_feature_pool,
                cat_feature_pool=cat_feature_pool,
                cv_splits=cv_splits,
                rf_fixed_params=rf_fixed_params,
                tuning_objective=tuning_objective,
                tuning_direction=tuning_direction,
                sampler_seed=sampler_seed,
                tune_trials=tune_trials,
                monotone_constraints=monotone_constraints,
            )
        )

    results_df = pd.DataFrame(rows)

    return results_df.reset_index(drop=True)


# execution code
topk_df = pd.read_csv(TOPK_PATH)

if topk_df.empty:
    raise ValueError("topk.csv is empty. Nothing to tune.")

tuned_results_df = run_feature_tuning(
    train_df=shared_train_df,
    target_col=TARGET_COL,
    combinations_df=topk_df,
    num_feature_pool=NUM_FEATURE_POOL,
    cat_feature_pool=CAT_FEATURE_POOL,
    cv_splits=shared_cv_splits,
    rf_fixed_params=RF_FIXED_PARAMS,
    tuning_objective=TUNING_OBJECTIVE,
    tuning_direction=TUNING_DIRECTION,
    sampler_seed=OPTUNA_SAMPLER_SEED,
    tune_trials=TUNE_TRIALS,
    monotone_constraints=MONOTONE_CONSTRAINTS,
)

print(f"Tuned combinations: {len(tuned_results_df)}")
display(tuned_results_df)

[I 2026-04-10 20:31:36,283] A new study created in memory with name: no-name-ac53fe36-8922-4caf-adcc-9964d4b3b6bb
[I 2026-04-10 20:33:41,617] Trial 0 finished with value: 6762.479391008344 and parameters: {'n_estimators': 350, 'max_depth': 16, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 0.40921304830970556, 'max_samples': 0.662397808134481}. Best is trial 0 with value: 6762.479391008344.
[I 2026-04-10 20:34:59,290] Trial 1 finished with value: 6950.807353819415 and parameters: {'n_estimators': 223, 'max_depth': 15, 'min_samples_split': 13, 'min_samples_leaf': 15, 'max_features': 0.3144091460070617, 'max_samples': 0.9879639408647978}. Best is trial 0 with value: 6762.479391008344.
[I 2026-04-10 20:36:33,908] Trial 2 finished with value: 6644.7924320418615 and parameters: {'n_estimators': 533, 'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.5129695700716763, 'max_samples': 0.8099025726528951}. Best is trial 2 with value: 6644.79243204

Tuned combinations: 5


,combination_id,groups,feature_list_json,feature_count,tuned_mean_rmse,tuned_std_rmse,tuned_mean_rmsle,tuned_std_rmsle,tuned_mean_r2,tuned_std_r2,...,tuned_fold_2_r2,tuned_fold_2_mape,tuned_fold_2_smape,tuned_fold_2_wmape,tuned_fold_3_rmse,tuned_fold_3_rmsle,tuned_fold_3_r2,tuned_fold_3_mape,tuned_fold_3_smape,tuned_fold_3_wmape
0,5,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9,6209.583364,1459.366524,1.507200,0.124911,0.892208,0.042650,...,0.931295,4377.576574,52.011778,21.852335,5423.968422,1.335715,0.912446,2637.115186,48.719900,26.205235
1,8,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",11,6494.890632,1720.451744,1.546542,0.159494,0.881211,0.054146,...,0.918994,8140.657978,52.811068,23.458876,5184.707861,1.336333,0.920000,2570.472438,48.225889,24.419743
2,2,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,6529.482293,1283.371220,1.754490,0.122511,0.882099,0.037596,...,0.914744,8665.391196,65.444537,28.677228,5734.882454,1.619895,0.902121,4556.393156,61.865406,31.433187
3,7,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",10,7181.375058,1778.390852,1.563299,0.150275,0.855651,0.060572,...,0.892776,7456.670475,53.090719,24.137888,5681.179051,1.357104,0.903945,2917.082806,49.153471,26.204777
4,3,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,7002.550478,1713.854328,1.533040,0.128303,0.862629,0.057109,...,0.911061,5028.950641,51.971845,22.817736,5957.182238,1.353900,0.894386,2802.435546,48.534005,27.825296


In [16]:
# =========================
# Stage 3 - save tuned results
# =========================
tuned_results_df.to_csv(TUNED_RESULTS_PATH, index=False)
print(f"Saved: {TUNED_RESULTS_PATH}")

Saved: results\random_forest_poisson\3_tuned_results.csv


In [17]:
# =========================
# Stage 4 - load tuned results, rank, select top 1
# =========================
tuned_results_df = pd.read_csv(TUNED_RESULTS_PATH)

if tuned_results_df.empty:
    raise ValueError("tuned_results.csv is empty. Cannot rank final candidates.")

sorted_tuned_results_df = sort_with_tolerances(
    df=tuned_results_df,
    sort_cols=TUNED_SORT_COLS,
    tolerances=TUNED_SORT_TOL,
).reset_index(drop=True)

display(sorted_tuned_results_df)

top1_df = sorted_tuned_results_df.head(1).reset_index(drop=True)

print(f"selected top {1} row out of top {K}")
display(top1_df)

,combination_id,groups,feature_list_json,feature_count,tuned_mean_rmse,tuned_std_rmse,tuned_mean_rmsle,tuned_std_rmsle,tuned_mean_r2,tuned_std_r2,...,tuned_fold_2_r2,tuned_fold_2_mape,tuned_fold_2_smape,tuned_fold_2_wmape,tuned_fold_3_rmse,tuned_fold_3_rmsle,tuned_fold_3_r2,tuned_fold_3_mape,tuned_fold_3_smape,tuned_fold_3_wmape
0,5,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9,6209.583364,1459.366524,1.507200,0.124911,0.892208,0.042650,...,0.931295,4377.576574,52.011778,21.852335,5423.968422,1.335715,0.912446,2637.115186,48.719900,26.205235
1,2,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,6529.482293,1283.371220,1.754490,0.122511,0.882099,0.037596,...,0.914744,8665.391196,65.444537,28.677228,5734.882454,1.619895,0.902121,4556.393156,61.865406,31.433187
2,8,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",11,6494.890632,1720.451744,1.546542,0.159494,0.881211,0.054146,...,0.918994,8140.657978,52.811068,23.458876,5184.707861,1.336333,0.920000,2570.472438,48.225889,24.419743
3,3,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",8,7002.550478,1713.854328,1.533040,0.128303,0.862629,0.057109,...,0.911061,5028.950641,51.971845,22.817736,5957.182238,1.353900,0.894386,2802.435546,48.534005,27.825296
4,7,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",10,7181.375058,1778.390852,1.563299,0.150275,0.855651,0.060572,...,0.892776,7456.670475,53.090719,24.137888,5681.179051,1.357104,0.903945,2917.082806,49.153471,26.204777


selected top 1 row out of top 5


,combination_id,groups,feature_list_json,feature_count,tuned_mean_rmse,tuned_std_rmse,tuned_mean_rmsle,tuned_std_rmsle,tuned_mean_r2,tuned_std_r2,...,tuned_fold_2_r2,tuned_fold_2_mape,tuned_fold_2_smape,tuned_fold_2_wmape,tuned_fold_3_rmse,tuned_fold_3_rmsle,tuned_fold_3_r2,tuned_fold_3_mape,tuned_fold_3_smape,tuned_fold_3_wmape
0,5,price_per_litre + customer + top_brand + flavo...,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",9,6209.583364,1459.366524,1.5072,0.124911,0.892208,0.04265,...,0.931295,4377.576574,52.011778,21.852335,5423.968422,1.335715,0.912446,2637.115186,48.7199,26.205235


In [18]:
# =========================
# Stage 4 - save best config
# =========================
top1_df.to_csv(BEST_CONFIG_PATH, index=False)
print(f"Saved: {BEST_CONFIG_PATH}")

Saved: results\random_forest_poisson\4_best_config.csv


In [19]:
# =========================
# Stage 5 - refit best on full train and evaluate test
# =========================
# methods
def callable_latency_ms_per_row(predict_fn, X_like, n_repeats=LATENCY_REPEATS):
    _ = predict_fn(X_like.iloc[: min(len(X_like), 256)].copy())
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        _ = predict_fn(X_like)
    elapsed = time.perf_counter() - t0
    return float((elapsed / n_repeats) * 1000.0 / max(len(X_like), 1))


def evaluate_predictions(
    *,
    model_name: str,
    tuning_method: str,
    model_objective: str,
    tuning_objective: str,
    column_list: Sequence[str],
    y_true: Sequence[float] | np.ndarray,
    y_pred: Sequence[float] | np.ndarray,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    series_cols: Sequence[str],
    target_col: str,
    time_col: str,
    tuning_time_s: float = 0.0,
    refit_time_s: float = 0.0,
    end_to_end_time_s: float = 0.0,
    inference_latency_ms_per_row: float = np.nan,
    model_size_bytes: float = np.nan,
    model_params: Mapping[str, object] | None = None,
    monotone_constraints: Mapping[str, int] | None = None,
):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 0.0, None)
    model_params = {} if model_params is None else model_params

    return {
        'Model': model_name,
        'Model Objective': model_objective,
        'Tuning Method': tuning_method,
        'Tuning Objective': tuning_objective,
        'column_list': json.dumps(list(column_list), ensure_ascii=True),
        'MAE': mae(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'RMSLE': rmsle(y_true, y_pred),
        'R2': r2(y_true, y_pred),
        'MAPE': mape_pct(y_true, y_pred),
        'sMAPE': smape_pct(y_true, y_pred),
        'wMAPE': wmape_pct(y_true, y_pred),
        'MASE': panel_mase(
            train_df=train_df,
            test_df=test_df,
            y_pred=y_pred,
            series_cols=series_cols,
            target_col=target_col,
            time_col=time_col,
            seasonality=SEASONALITY,
        ),
        'Tuning Time (s)': float(tuning_time_s),
        'Refit Time (s)': float(refit_time_s),
        'End-to-End Time (s)': float(end_to_end_time_s),
        'Inference Latency (ms/row)': float(inference_latency_ms_per_row),
        'Model Size (bytes)': float(model_size_bytes),
        'Model Params (json)': json.dumps(model_params, ensure_ascii=True),
        'Monotone Constraints Config (json)': json.dumps(monotone_constraints, ensure_ascii=True),
    }

def serialized_size_bytes(obj):
    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix='.joblib', delete=False) as tmp:
            tmp_path = tmp.name
        joblib.dump(obj, tmp_path)
        return int(os.path.getsize(tmp_path))
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)


# execution code
top1_df = pd.read_csv(BEST_CONFIG_PATH)

if top1_df.empty:
    raise ValueError("best_config.csv is empty. Cannot run final test evaluation.")

if len(top1_df) != 1:
    raise ValueError(f"best_config.csv must contain exactly 1 row, got {len(top1_df)}.")


best_row = top1_df.iloc[0]
feature_cols = json.loads(best_row["feature_list_json"])
best_params = json.loads(best_row["best_params_json"])
tuning_time_s = float(best_row["tuning_time_s"])

missing_train_features = [col for col in feature_cols if col not in shared_train_df.columns]
missing_test_features = [col for col in feature_cols if col not in test_df.columns]
if missing_train_features:
    raise ValueError(f"Selected features are missing from shared_train_df: {missing_train_features}")
if missing_test_features:
    raise ValueError(f"Selected features are missing from test_df: {missing_test_features}")

num_cols, cat_cols = split_feature_types(
    feature_cols,
    num_feature_pool=NUM_FEATURE_POOL,
    cat_feature_pool=CAT_FEATURE_POOL,
)

X_train = shared_train_df.loc[:, feature_cols].copy()
y_train = shared_train_df[TARGET_COL].to_numpy(dtype=float)
X_test = test_df.loc[:, feature_cols].copy()
y_test = test_df[TARGET_COL].to_numpy(dtype=float)


model_params = {
    **best_params,
    "monotonic_cst": get_monotone_constraints(
        X_fit=X_train,
        monotone_constraints=MONOTONE_CONSTRAINTS,
        num_cols=num_cols,
        cat_cols=cat_cols,
    ).tolist(),
}

best_model = Pipeline(
    steps=[
        ("prep", make_preprocessor(num_cols, cat_cols)),
        ("model", RandomForestRegressor(**model_params)),
    ]
)

t0 = time.perf_counter()
best_model.fit(X_train, y_train)
refit_time_s = time.perf_counter() - t0

y_pred = best_model.predict(X_test)
y_pred = np.clip(y_pred, 0.0, None)
inference_latency_ms_per_row = callable_latency_ms_per_row(best_model.predict, X_test)
model_size_bytes = serialized_size_bytes(best_model)
end_to_end_time_s = tuning_time_s + refit_time_s

metrics_row = evaluate_predictions(
    model_name="Random Forest",
    tuning_method="Optuna TPE",
    model_objective=MODEL_OBJECTIVE,
    tuning_objective=TUNING_OBJECTIVE,
    column_list=feature_cols,
    y_true=y_test,
    y_pred=y_pred,
    train_df=shared_train_df,
    test_df=test_df,
    series_cols=SERIES_COLS,
    target_col=TARGET_COL,
    time_col=TIME_COL,
    tuning_time_s=tuning_time_s,
    refit_time_s=refit_time_s,
    end_to_end_time_s=end_to_end_time_s,
    inference_latency_ms_per_row=inference_latency_ms_per_row,
    model_size_bytes=model_size_bytes,
    model_params=model_params,
    monotone_constraints=MONOTONE_CONSTRAINTS,
)
final_metrics_df = pd.DataFrame([metrics_row])

final_predictions_df = test_df.copy()
final_predictions_df["y_true"] = y_test
final_predictions_df["y_pred"] = y_pred

display(final_metrics_df)
display(final_predictions_df.head())


,Model,Model Objective,Tuning Method,Tuning Objective,column_list,MAE,RMSE,RMSLE,R2,MAPE,sMAPE,wMAPE,MASE,Tuning Time (s),Refit Time (s),End-to-End Time (s),Inference Latency (ms/row),Model Size (bytes),Model Params (json),Monotone Constraints Config (json)
0,Random Forest,poisson,Optuna TPE,mean_rmse,"[""price_per_litre"", ""customer"", ""top_brand"", ""...",2226.387407,6575.118419,1.533052,0.869978,4808.390186,52.112361,29.940928,33.527571,1136.142512,195.861732,1332.004244,0.027496,227182954.0,"{""n_estimators"": 439, ""max_depth"": 16, ""min_sa...",{}


,product_sku_code,customer,year,yearweek,nielsen_total_volume,promotion_indicator,top_brand,flavor_internal,pack_type_internal,pack_size_internal,...,price_per_item,_price_before_impute,price_per_litre,price_per_100ml,week,week_sin,week_cos,continuous_week,y_true,y_pred
73887,201050,L2_ASDA,2025,202515,387,0,COCA-COLA,COLA,GLASS,250,...,4.6970,4.6970,4.697000,0.469700,15,0.970942,-0.239316,118,387.0,409.687924
73888,201050,L2_MORRISONS,2025,202515,608,0,COCA-COLA,COLA,GLASS,250,...,4.6955,4.6955,4.695500,0.469550,15,0.970942,-0.239316,118,608.0,552.750182
73889,201050,L2_SAINSBURY'S,2025,202515,1781,0,COCA-COLA,COLA,GLASS,250,...,4.5156,4.5156,4.515600,0.451560,15,0.970942,-0.239316,118,1781.0,1194.187441
73890,201051,L2_SAINSBURY'S,2025,202515,805,0,COCA-COLA LIGHT / DIET COKE,COLA,GLASS,250,...,3.9752,3.9752,3.975200,0.397520,15,0.970942,-0.239316,118,805.0,793.448069
73891,207883,L2_ASDA,2025,202515,1230,1,APPLETISER,APPLE,CAN,150,...,0.5231,0.5231,3.487333,0.348733,15,0.970942,-0.239316,118,1230.0,1078.620659


In [20]:
# =========================
# Stage 5 - save final results
# =========================
final_metrics_df.to_csv(TEST_METRICS_PATH, index=False)
final_predictions_df.to_csv(TEST_PREDICTIONS_PATH, index=False)
print(f'Saved: {TEST_METRICS_PATH}')
print(f'Saved: {TEST_PREDICTIONS_PATH}')

Saved: results\random_forest_poisson\5_test_metrics.csv
Saved: results\random_forest_poisson\5_test_predictions.csv


In [21]:
# =========================
# Stage 5 - save final model
# =========================
joblib.dump(best_model, BEST_MODEL_PATH)
print(f'Saved: {BEST_MODEL_PATH}')

Saved: results\random_forest_poisson\best_model.joblib
